# Statistics of a run

Runs measure the power spectra of the maps (NaMaster) and the 2pt functions of the catalogs (TreeCorr), saved in [SACC](https://github.com/LSSTDESC/sacc) files, and plot them against the predictions of CCL in the `plots/` folder of each product. This notebook reads the measurements, shows the plots of the pipeline, and compares the runs without and with baryonification.

In [ ]:
import os

# be fair to others on a login node, before importing numba and OpenMP libraries
os.environ['OMP_NUM_THREADS'] = os.environ['NUMBA_NUM_THREADS'] = '8'

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

from pollux import PolluxLoader

# a realization of the validation run (scripts/run_validation.yaml of the shared space), in your scratch
realization_dir = os.path.expandvars('$PSCRATCH/HACC-HOS-Y1/validation/HACC091_1024/nside2048/seed0')
baryon_tag = 'S19_Mc1e14h_thetaej4.0_proj_cutoff'
catalog_tag = 'poisson_b2.0_nbar1.5_lognormal_bias10pc'

dmo = PolluxLoader(realization_dir, catalog_tag=catalog_tag)
dmb = PolluxLoader(realization_dir, baryon_tag=baryon_tag, catalog_tag=catalog_tag)

## Convergence power spectra

The SACC file of the convergence maps has the power spectra of all pairs of tomographic bins. Below, the ratio of the auto-spectra with and without baryonification: the suppression of power at small scales.

In [ ]:
cls = {name: loader.load_kappa_cls() for name, loader in (('dmo', dmo), ('dmb', dmb))}

for ibin in range(5):
    ell, cl_dmo = cls['dmo'].get_ell_cl('galaxy_convergence_cl', f'kappa_{ibin}', f'kappa_{ibin}')
    _, cl_dmb = cls['dmb'].get_ell_cl('galaxy_convergence_cl', f'kappa_{ibin}', f'kappa_{ibin}')
    plt.semilogx(ell, cl_dmb / cl_dmo - 1, label=f'bin {ibin}')
plt.axhline(0, color='k', lw=0.5)
plt.gca().set(xlabel=r'$\ell$', ylabel=r'$C_\ell^{\rm dmb} / C_\ell^{\rm dmo} - 1$')
plt.legend();

## Catalog 2pt functions

The SACC file of the catalogs has the galaxy clustering $w(\theta)$ (tracers `density_i`), galaxy-galaxy lensing $\gamma_t(\theta)$ and cosmic shear $\xi_\pm(\theta)$ (tracers `kappa_i`), with shot-noise error bars.

In [ ]:
xi = {name: loader.load_catalog_2pt() for name, loader in (('dmo', dmo), ('dmb', dmb))}
print(xi['dmb'].get_data_types())

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ibin in range(5):
    theta, w = xi['dmo'].get_theta_xi('galaxy_density_xi', f'density_{ibin}', f'density_{ibin}')
    axes[0].loglog(theta, theta * w, label=f'bin {ibin}')
    theta, gt = xi['dmo'].get_theta_xi('galaxy_shearDensity_xi_t', f'density_{ibin}', 'kappa_4')
    axes[1].loglog(theta, theta * gt)
    theta, xip = xi['dmo'].get_theta_xi('galaxy_shear_xi_plus', f'kappa_{ibin}', f'kappa_{ibin}')
    axes[2].loglog(theta, theta * xip)
for ax, label in zip(axes, [r'$\theta w(\theta)$', r'$\theta \gamma_t(\theta)$, sources in bin 4', r'$\theta \xi_+(\theta)$']):
    ax.set(xlabel=r'$\theta$ (arcmin)', ylabel=label)
axes[0].legend()
fig.tight_layout()

Baryons suppress $\xi_+$ at small scales, more for nearby sources:

In [ ]:
for ibin in range(5):
    theta, xip_dmo = xi['dmo'].get_theta_xi('galaxy_shear_xi_plus', f'kappa_{ibin}', f'kappa_{ibin}')
    _, xip_dmb = xi['dmb'].get_theta_xi('galaxy_shear_xi_plus', f'kappa_{ibin}', f'kappa_{ibin}')
    plt.semilogx(theta, xip_dmb / xip_dmo - 1, 'o-', label=f'bin {ibin}')
plt.axhline(0, color='k', lw=0.5)
plt.gca().set(xlabel=r'$\theta$ (arcmin)', ylabel=r'$\xi_+^{\rm dmb} / \xi_+^{\rm dmo} - 1$', ylim=(-0.3, 0.1))
plt.legend();

## Comparison to theory

The pipeline plots each measurement against the CCL prediction made with the same tracers, smoothing and (for `dmb`) the matching baryonic correction of the matter power spectrum. The dotted lines are the reference scales: pixel size and smoothing of the galaxy density. Each figure is in the `plots/` folder of its product, here the shear 2pt functions of the baryonified catalogs, as ratios to the prediction.

In [ ]:
from IPython.display import Image

Image(os.path.join(dmb.catalog_dir(), 'plots', 'xi_shear_ratio.png'), width=900)

In [ ]:
Image(os.path.join(dmb.catalog_dir(), 'plots', 'xi_density_ratio.png'), width=900)

The same functions with baryons, relative to those without (points), against the ratio of the predictions (dashed):

In [ ]:
Image(os.path.join(dmb.catalog_dir(), 'plots', 'xi_shear_ratio_to_reference.png'), width=900)